In [1]:
import os

PROJECT_DIR = ""

os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/figures", exist_ok=True)


In [ ]:
%pip install -q datasets transformers torch pandas

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the official SST-2 dataset
dataset = load_dataset("stanfordnlp/sst2")

# Inspect the available splits
print(dataset)

# Display five validation examples
validation_df = pd.DataFrame(dataset["validation"][:5])

validation_df["label_name"] = validation_df["label"].map({
    0: "negative",
    1: "positive"
})

display(validation_df[["sentence", "label_name"]])

README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

... (2 outputs left out)

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


,sentence,label_name
0,it 's a charming and often affecting journey .,positive
1,unflinchingly bleak and desperate,negative
2,allows us to hope that nolan is poised to emba...,positive
3,"the acting , costumes , music , cinematography...",positive
4,"it 's slow -- very , very slow .",negative


In [ ]:
from transformers import pipeline

MODEL_ID = (
    "distilbert/"
    "distilbert-base-uncased-finetuned-sst-2-english"
)

classifier = pipeline(
    "text-classification",
    model=MODEL_ID,
    top_k=None
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
test_texts = [
    "The movie was wonderful.",
    "The movie was terrible."
]

predictions = classifier(test_texts)

for text, scores in zip(test_texts, predictions):
    print(f"\nSentence: {text}")

    for result in scores:
        print(
            f"  {result['label']}: "
            f"{result['score']:.4f}"
        )


Sentence: The movie was okay.
  POSITIVE: 0.9998
  NEGATIVE: 0.0002

Sentence: The movie was terrible.
  NEGATIVE: 0.9998
  POSITIVE: 0.0002


In [ ]:
import pandas as pd

# Convert the validation split into a DataFrame
validation_df = pd.DataFrame(dataset["validation"])

# Add readable sentiment labels
validation_df["label_name"] = validation_df["label"].map({
    0: "negative",
    1: "positive"
})

# Calculate sentence length
validation_df["word_count"] = (
    validation_df["sentence"].str.split().str.len()
)

# Keep relatively short sentences for easier controlled edits
candidates = validation_df[
    validation_df["word_count"].between(4, 12)
]

# Display 10 positive and 10 negative candidates
for sentiment, label in [("Positive", 1), ("Negative", 0)]:
    print(f"\n{sentiment} candidates:")

    sample = (
        candidates[candidates["label"] == label]
        .sample(n=10, random_state=42)
    )

    display(sample[["sentence", "label_name", "word_count"]])


Positive candidates:


,sentence,label_name,word_count
773,it 's a beautiful madness .,positive,6
741,but it still jingles in the pocket .,positive,8
679,filmmakers who can deftly change moods are tre...,positive,12
848,his healthy sense of satire is light and fun ...,positive,10
306,the story and structure are well-honed .,positive,7
746,people cinema at its finest .,positive,6
643,"the jabs it employs are short , carefully plac...",positive,12
383,a warm but realistic meditation on friendship ...,positive,12
0,it 's a charming and often affecting journey .,positive,9
112,hilariously inept and ridiculous .,positive,5



Negative candidates:


,sentence,label_name,word_count
631,burns never really harnesses to full effect th...,negative,11
58,manages to be both repulsively sadistic and mu...,negative,9
18,... the movie is just a plain old monster .,negative,10
650,"too slow , too long and too little happens .",negative,10
539,do not see this film .,negative,6
578,just not campy enough,negative,4
256,rarely has so much money delivered so little e...,negative,10
377,nothing is sacred in this gut-buster .,negative,7
751,one long string of cliches .,negative,6
74,a string of rehashed sight gags based in insip...,negative,11


In [ ]:
test_pairs = [
    (
        "The movie was okay.",
        "The movie was not okay."
    ),
    (
        "The story and structure are well-honed.",
        "The story and structure are not well-honed."
    ),
    (
        "The acting was excellent.",
        "The acting was not excellent."
    ),
]

for original, modified in test_pairs:
    results = classifier([original, modified])

    def get_scores(scores):
        return {
            item["label"]: item["score"]
            for item in scores
        }

    original_scores = get_scores(results[0])
    modified_scores = get_scores(results[1])

    print(f"\nOriginal: {original}")
    print(f"Modified: {modified}")

    print(
        "Original positive probability:",
        round(original_scores["POSITIVE"], 4)
    )
    print(
        "Modified positive probability:",
        round(modified_scores["POSITIVE"], 4)
    )
    print(
        "Change in positive probability:",
        round(
            modified_scores["POSITIVE"]
            - original_scores["POSITIVE"],
            4
        )
    )


Original: The movie was okay.
Modified: The movie was not okay.
Original positive probability: 0.9998
Modified positive probability: 0.0002
Change in positive probability: -0.9996

Original: The story and structure are well-honed.
Modified: The story and structure are not well-honed.
Original positive probability: 0.9998
Modified positive probability: 0.0003
Change in positive probability: -0.9995

Original: The acting was excellent.
Modified: The acting was not excellent.
Original positive probability: 0.9999
Modified positive probability: 0.0003
Change in positive probability: -0.9996
